In [1]:
# 04_two_stage_mlp_wj_rerank.ipynb
# Two-stage learned retrieval:
#   MLP 512-d cosine HNSW candidate generation + exact WeightedJaccard rerank
#
# This tests larger candidate pools than 02_mlp_cosine.ipynb, so we can see
# whether the MLP is good enough as a cheap candidate generator.


In [2]:
import gc
import os
import pickle
import time

import nmslib
import numpy as np
import psutil
import torch
import torch.nn as nn
import torch.nn.functional as F
from tqdm import tqdm


THREADS = 32
QUERY_START_10K = 8000
QUERY_START_FULL = 187019


class QuadtreeCompressorV1(nn.Module):
    def __init__(self, in_dim, out_dim=512):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, 4096, bias=False), nn.BatchNorm1d(4096), nn.ReLU(),
            nn.Linear(4096, 1024, bias=False), nn.BatchNorm1d(1024), nn.ReLU(),
            nn.Linear(1024, out_dim, bias=False), nn.BatchNorm1d(out_dim),
        )

    def forward(self, x):
        return self.net(x)


class QuadtreeCompressorV1Fixed(nn.Module):
    def __init__(self, in_dim, out_dim=512):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, 4096, bias=False), nn.BatchNorm1d(4096), nn.ReLU(),
            nn.Linear(4096, 1024, bias=False), nn.BatchNorm1d(1024), nn.ReLU(),
            nn.Linear(1024, out_dim, bias=False), nn.BatchNorm1d(out_dim),
        )

    def forward(self, x):
        x = torch.log1p(x * 1e6)
        return self.net(x)


def get_mem_mb():
    return psutil.Process(os.getpid()).memory_info().rss / 1024**2


def recall_at_k(gt_lookup, nbrs, query_start_id, k):
    total = 0.0
    count = 0
    for i, (ids, _) in enumerate(nbrs):
        qid = query_start_id + i
        gt = set(gt_lookup.get(qid, [])[:k])
        if not gt:
            continue
        total += len(gt & set(ids[:k])) / len(gt)
        count += 1
    return total / count if count else 0.0


def eval_recall(gt_lookup, nbrs, query_start_id, max_k):
    return {
        k: recall_at_k(gt_lookup, nbrs, query_start_id, k)
        for k in (10, 50, 100, 500)
        if k <= max_k
    }


def generate_embeddings(model, data, device, batch_size=512):
    model.eval()
    chunks = []
    with torch.no_grad():
        for start in tqdm(range(0, len(data), batch_size), desc="Embedding"):
            batch = torch.tensor(data[start:start + batch_size],
                                 dtype=torch.float32,
                                 device=device)
            chunks.append(F.normalize(model(batch), dim=1).cpu().numpy())
    return np.vstack(chunks)


def build_cosine_index(corpus_embs):
    m0 = get_mem_mb()
    idx = nmslib.init(method="hnsw", space="cosinesimil")
    for i in tqdm(range(len(corpus_embs)), desc="Adding", mininterval=2.0):
        idx.addDataPoint(i, corpus_embs[i])
    t0 = time.time()
    idx.createIndex({"M": 20, "efConstruction": 200, "post": 1},
                    print_progress=True)
    build_s = time.time() - t0
    idx_mb = get_mem_mb() - m0
    idx.setQueryTimeParams({"efSearch": 200})
    return idx, build_s, idx_mb


def rerank_wj_dense(query_qt, nbrs_raw, corpus_qt, corpus_sums):
    """Exact CPU rerank with one dense min pass.

    Weighted Jaccard uses sum(min) / sum(max). For nonnegative vectors,
    sum(max) = sum(q) + sum(c) - sum(min), so we avoid the second full
    K x dim maximum allocation used by the original implementation.
    """
    query_sums = query_qt.sum(axis=1)
    reranked = []
    for i, (ids, _) in enumerate(tqdm(nbrs_raw, desc="CPU WJ rerank")):
        ids = np.asarray(ids, dtype=np.int64)
        q_vec = query_qt[i]
        c_vecs = corpus_qt[ids]
        mins = np.minimum(q_vec, c_vecs).sum(axis=1)
        maxs = query_sums[i] + corpus_sums[ids] - mins
        wj = mins / np.maximum(maxs, 1e-10)
        order = np.argsort(-wj)
        reranked.append((ids[order].tolist(), []))
    return reranked


def rerank_wj_gpu(query_qt, nbrs_raw, corpus_qt, corpus_sums, device, batch_size=16):
    """Exact GPU rerank; handles variable-length nmslib candidate lists."""
    corpus_t = torch.from_numpy(corpus_qt).to(device=device, dtype=torch.float32)
    corpus_sums_t = torch.from_numpy(corpus_sums).to(device=device, dtype=torch.float32)
    reranked = [None] * len(nbrs_raw)

    for start in tqdm(range(0, len(nbrs_raw), batch_size), desc="GPU WJ rerank"):
        batch = nbrs_raw[start:start + batch_size]
        groups = {}

        for offset, (ids, _) in enumerate(batch):
            ids_arr = np.asarray(ids, dtype=np.int64)
            groups.setdefault(len(ids_arr), []).append((start + offset, ids_arr))

        for cand_len, items in groups.items():
            if cand_len == 0:
                for absolute_i, _ in items:
                    reranked[absolute_i] = ([], [])
                continue

            ids_np = np.stack([ids for _, ids in items], axis=0)
            query_np = np.stack(
                [query_qt[absolute_i] for absolute_i, _ in items],
                axis=0,
            )

            ids_t = torch.from_numpy(ids_np).to(device=device)
            q_t = torch.from_numpy(query_np).to(device=device, dtype=torch.float32)

            c_t = corpus_t[ids_t]
            mins = torch.minimum(q_t[:, None, :], c_t).sum(dim=2)
            maxs = q_t.sum(dim=1, keepdim=True) + corpus_sums_t[ids_t] - mins
            wj = mins / maxs.clamp_min(1e-10)
            order = torch.argsort(wj, dim=1, descending=True).cpu().numpy()

            for row, (absolute_i, ids) in zip(order, items):
                reranked[absolute_i] = (ids[row].tolist(), [])

    del corpus_t, corpus_sums_t
    if device.type == "cuda":
        torch.cuda.empty_cache()

    return reranked


def rerank_wj(query_qt, nbrs_raw, corpus_qt, corpus_sums, mode, device, batch_size):
    if mode == "dense":
        return rerank_wj_dense(query_qt, nbrs_raw, corpus_qt, corpus_sums)
    if mode == "gpu":
        return rerank_wj_gpu(query_qt, nbrs_raw, corpus_qt, corpus_sums,
                             device, batch_size=batch_size)
    raise ValueError(f"unknown rerank mode: {mode}")


def run_dataset(name, device, candidate_ks, rerank_mode="dense", rerank_batch_size=16):
    if name == "10k":
        qt = np.load("/tmp/qt_10k.npy")
        with open("/tmp/gt_lookup_10k.pkl", "rb") as f:
            gt = pickle.load(f)
        query_start = QUERY_START_10K
        model = QuadtreeCompressorV1(qt.shape[1]).to(device)
        model.load_state_dict(torch.load("/tmp/best_compressor_v1_clean.pt",
                                         weights_only=True,
                                         map_location=device))
    elif name == "full":
        qt = np.load("/tmp/qtree_vectors_full.npy")
        with open("/tmp/gt_lookup_full.pkl", "rb") as f:
            gt = pickle.load(f)
        query_start = QUERY_START_FULL
        model = QuadtreeCompressorV1Fixed(qt.shape[1]).to(device)
        model.load_state_dict(torch.load("/tmp/best_compressor_full_fixed.pt",
                                         weights_only=True,
                                         map_location=device))
    else:
        raise ValueError(f"unknown dataset: {name}")

    corpus_qt = qt[:query_start]
    query_qt = qt[query_start:]
    corpus_sums = corpus_qt.sum(axis=1)

    print(f"\n{'=' * 72}")
    print(f"TWO-STAGE MLP-COSINE CANDIDATES + EXACT WJ RERANK -- {name}")
    print(f"{'=' * 72}")
    print(f"corpus={corpus_qt.shape} | queries={query_qt.shape}")

    embs = generate_embeddings(model, qt, device)
    corpus_embs = embs[:query_start]
    query_embs = embs[query_start:]
    vec_mb = corpus_embs.nbytes / 1024**2
    print(f"embeddings={embs.shape} | Vec={vec_mb:.1f} MB")

    idx, build_s, idx_mb = build_cosine_index(corpus_embs)
    print(f"Build={build_s:.1f}s | Index mem={idx_mb:.1f} MB")

    results = {}
    for k in candidate_ks:
        print(f"\nCandidate K={k}")
        t0 = time.time()
        nbrs_raw = idx.knnQueryBatch(query_embs, k=k, num_threads=THREADS)
        hnsw_s = time.time() - t0

        t0 = time.time()
        nbrs_rr = rerank_wj(query_qt, nbrs_raw, corpus_qt, corpus_sums,
                            rerank_mode, device, rerank_batch_size)
        rerank_s = time.time() - t0

        total_s = hnsw_s + rerank_s
        qps = len(query_embs) / total_s
        rec = eval_recall(gt, nbrs_rr, query_start, max_k=k)
        results[f"k{k}_wj_rerank"] = {
            **rec,
            "candidate_k": k,
            "rerank_mode": rerank_mode,
            "qps": qps,
            "hnsw_s": hnsw_s,
            "rerank_s": rerank_s,
            "build_s": build_s,
            "vec_mb": vec_mb,
            "idx_mb": idx_mb,
        }

        print(f"HNSW={hnsw_s:.2f}s | rerank={rerank_s:.2f}s | QPS={qps:.1f}")
        for kk, rr in rec.items():
            print(f"  R@{kk:<4} = {rr:.4f}")

        del nbrs_raw, nbrs_rr
        gc.collect()

    return results


In [3]:
# Configuration
# Start with dataset = "10k" for a quick run. Use "full" after validating.
dataset = "10k"       # "10k", "full", or "both"
device = torch.device("cuda:0")
candidate_ks = [500, 1000, 2000]

# Rerank modes:
#   "dense" = exact CPU rerank, optimized to avoid the full max() allocation
#   "gpu"   = exact GPU rerank, fastest when the original corpus fits on GPU
rerank_mode = "gpu"
rerank_batch_size = 32

out_path = "/tmp/results_two_stage_mlp_wj.pkl"


In [4]:
# Run experiment
all_results = {}
datasets = ["10k", "full"] if dataset == "both" else [dataset]
for name in datasets:
    all_results[name] = run_dataset(name, device, candidate_ks,
                                    rerank_mode=rerank_mode,
                                    rerank_batch_size=rerank_batch_size)

with open(out_path, "wb") as f:
    pickle.dump(all_results, f)
print(f"\nSaved results to {out_path}")



TWO-STAGE MLP-COSINE CANDIDATES + EXACT WJ RERANK -- 10k
corpus=(8000, 18499) | queries=(2000, 18499)


Embedding: 100%|██████████| 20/20 [00:00<00:00, 64.01it/s]


embeddings=(10000, 512) | Vec=15.6 MB


Adding: 100%|██████████| 8000/8000 [00:00<00:00, 686563.79it/s]

0%   10   20   30   40   50   60   70   80   90   100%
|----|----|----|----|----|----|----|----|----|----|
***************************************************

0%   10   20   30   40   50   60   70   80   90   100%
|----|----|----|----|----|----|----|----|----|----|
***************************************************
*

Build=0.2s | Index mem=32.8 MB

Candidate K=500


...

HNSW=0.08s | rerank=0.61s | QPS=2903.5
  R@10   = 0.9966
  R@50   = 0.9984
  R@100  = 0.9980
  R@500  = 0.9540

Candidate K=1000


GPU WJ rerank: 100%|██████████| 63/63 [00:00<00:00, 65.51it/s]

HNSW=0.11s | rerank=1.02s | QPS=1766.2
  R@10   = 0.9966
  R@50   = 0.9985
  R@100  = 0.9987
  R@500  = 0.9825

Candidate K=2000


GPU WJ rerank: 100%|██████████| 63/63 [00:00<00:00, 65.65it/s]


HNSW=0.11s | rerank=1.02s | QPS=1763.9
  R@10   = 0.9966
  R@50   = 0.9985
  R@100  = 0.9987
  R@500  = 0.9825

Saved results to /tmp/results_two_stage_mlp_wj.pkl


In [5]:
# Print compact summary from saved results
with open(out_path, "rb") as f:
    saved = pickle.load(f)

print(f"\n{'=' * 110}")
print("TWO-STAGE MLP-COSINE + EXACT WJ RERANK SUMMARY")
print(f"{'=' * 110}")
print(f"{'Dataset':<8} {'Method':<22} {'R@10':>7} {'R@50':>7} {'R@100':>7} {'R@500':>7} "
      f"{'QPS':>8} {'HNSW(s)':>9} {'WJ(s)':>9} {'Build':>8} {'Vec(MB)':>9} {'Idx(MB)':>9}")
print("-" * 110)

for ds_name, ds_results in saved.items():
    for key, res in ds_results.items():
        def fmt(k):
            return f"{res[k]:>7.4f}" if k in res else f"{'-':>7}"
        print(f"{ds_name:<8} {key:<22} {fmt(10)} {fmt(50)} {fmt(100)} {fmt(500)} "
              f"{res['qps']:>8.1f} {res['hnsw_s']:>9.2f} {res['rerank_s']:>9.2f} "
              f"{res['build_s']:>7.1f}s {res['vec_mb']:>9.1f} {res['idx_mb']:>9.1f}")



TWO-STAGE MLP-COSINE + EXACT WJ RERANK SUMMARY
Dataset  Method                    R@10    R@50   R@100   R@500      QPS   HNSW(s)     WJ(s)    Build   Vec(MB)   Idx(MB)
--------------------------------------------------------------------------------------------------------------
10k      k500_wj_rerank          0.9966  0.9984  0.9980  0.9540   2903.5      0.08      0.61     0.2s      15.6      32.8
10k      k1000_wj_rerank         0.9966  0.9985  0.9987  0.9825   1766.2      0.11      1.02     0.2s      15.6      32.8
10k      k2000_wj_rerank         0.9966  0.9985  0.9987  0.9825   1763.9      0.11      1.02     0.2s      15.6      32.8


In [6]:
# Show all saved two-stage results from the pickle
import pickle
from pathlib import Path

results_path = Path(out_path)
print(f"Loading: {results_path}")

with open(results_path, "rb") as f:
    saved = pickle.load(f)

# Support both the original one-run format and the newer appended-runs format.
if "runs" not in saved:
    saved = {"runs": {"legacy": {"config": {}, "datasets": saved}}}

rows = []
for run_name, run in saved["runs"].items():
    config = run.get("config", {})
    for ds_name, ds_results in run.get("datasets", {}).items():
        for method, res in ds_results.items():
            rows.append({
                "run": run_name,
                "dataset": ds_name,
                "method": method,
                "mode": res.get("rerank_mode", config.get("rerank_mode", "-")),
                "candidate_k": res.get("candidate_k", "-"),
                "R@10": res.get(10, float("nan")),
                "R@50": res.get(50, float("nan")),
                "R@100": res.get(100, float("nan")),
                "R@500": res.get(500, float("nan")),
                "qps": res.get("qps", float("nan")),
                "hnsw_s": res.get("hnsw_s", float("nan")),
                "rerank_s": res.get("rerank_s", float("nan")),
                "build_s": res.get("build_s", float("nan")),
                "vec_mb": res.get("vec_mb", float("nan")),
                "idx_mb": res.get("idx_mb", float("nan")),
            })

print(f"Saved runs: {len(saved['runs'])} | result rows: {len(rows)}")
print(f"\n{'=' * 152}")
print("ALL TWO-STAGE MLP-COSINE + WJ RERANK RESULTS")
print(f"{'=' * 152}")
print(f"{'Run':<30} {'Data':<6} {'Mode':<6} {'K':>5} {'R@10':>7} {'R@50':>7} {'R@100':>7} {'R@500':>7} "
      f"{'QPS':>9} {'HNSW(s)':>9} {'WJ(s)':>9} {'Build':>8} {'Vec(MB)':>9} {'Idx(MB)':>9}")
print("-" * 152)

for row in rows:
    def fmt_recall(key):
        val = row[key]
        return f"{val:>7.4f}" if val == val else f"{'-':>7}"

    print(f"{row['run']:<30} {row['dataset']:<6} {row['mode']:<6} {row['candidate_k']:>5} "
          f"{fmt_recall('R@10')} {fmt_recall('R@50')} {fmt_recall('R@100')} {fmt_recall('R@500')} "
          f"{row['qps']:>9.1f} {row['hnsw_s']:>9.2f} {row['rerank_s']:>9.2f} "
          f"{row['build_s']:>7.1f}s {row['vec_mb']:>9.1f} {row['idx_mb']:>9.1f}")

# Also expose rows for optional ad-hoc analysis in later cells.
all_saved_rows = rows


Loading: /tmp/results_two_stage_mlp_wj.pkl
Saved runs: 1 | result rows: 3

ALL TWO-STAGE MLP-COSINE + WJ RERANK RESULTS
Run                            Data   Mode       K    R@10    R@50   R@100   R@500       QPS   HNSW(s)     WJ(s)    Build   Vec(MB)   Idx(MB)
--------------------------------------------------------------------------------------------------------------------------------------------------------
legacy                         10k    gpu      500  0.9966  0.9984  0.9980  0.9540    2903.5      0.08      0.61     0.2s      15.6      32.8
legacy                         10k    gpu     1000  0.9966  0.9985  0.9987  0.9825    1766.2      0.11      1.02     0.2s      15.6      32.8
legacy                         10k    gpu     2000  0.9966  0.9985  0.9987  0.9825    1763.9      0.11      1.02     0.2s      15.6      32.8
